(La Interfaz Unificada), necesitamos un escenario del mundo real. Imaginemos que trabajas en el área de Gobierno de Datos y Cumplimiento de una organización financiera y necesitas un asistente automatizado que analice reglas de calidad de datos y reportes regulatorios.
En este ejemplo completo e integral, construiremos la base de conocimiento (RAG) con políticas normativas reales y luego ejecutaremos la misma cadena usando Batch (para procesar auditorías masivas), Stream (para el chatbot de los usuarios) y Async (para montar un servicio web rápido).

In [15]:
import asyncio
from dotenv import load_dotenv

¿Qué es? El "Director de Orquesta Multitarea".

¿Para qué sirve? Por defecto, Python es un trabajador que solo sabe hacer una cosa a la vez: si está cocinando, no puede contestar el teléfono hasta que termine de cocinar. asyncio es una librería que le enseña a Python a ser multitarea. Le permite decir: "Ok, puse el agua a hervir (llamar a Gemini), mientras el agua hierve y responde, voy a avanzar limpiando la mesa (atender otra consulta)".

Si no estuviera: Tu programa se congelaría por completo cada vez que le pida algo a Google, y nadie más podría usarlo hasta que Google responda.

2. from dotenv import load_dotenv
¿Qué es? El "Guardia de Seguridad del Búnker".

¿Para qué sirve? Se encarga de buscar un archivo oculto en tu carpeta llamado .env donde guardas tu API Key (tu contraseña secreta de Google). Su trabajo es leer esa contraseña y ponerla en la memoria del sistema de forma oculta.

Si no estuviera: Tendrías que escribir tu contraseña real en texto plano dentro del código. Si subes ese código a GitHub o se lo pasas a un amigo, te robarían la contraseña y podrían gastar dinero a tu nombre.

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.prompts import ChatPromptTemplate
¿Qué es? El "Molde Industrial para Ladrillos".
¿Para qué sirve? Los modelos de IA necesitan instrucciones claras. En lugar de escribir un texto a mano cada vez, esta herramienta te permite crear una plantilla estructurada con "espacios en blanco" (como {pregunta}) para que el sistema los rellene automáticamente con los datos del usuario.
Si no estuviera: Tendrías que usar comandos de texto manuales muy complejos para pegar cadenas de texto una detrás de otra, lo cual suele llenar el código de errores de tipeo.

4. from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
Aquí estamos trayendo dos herramientas del mismo camión de Google:
ChatGoogleGenerativeAI (El "Cerebro Escritor"): Es el puente directo con Gemini. Es el que recibe un texto, lo razona, y redacta una respuesta humana.

GoogleGenerativeAIEmbeddings (El "Traductor Matemático"): Las computadoras no entienden letras. Este componente toma una frase humana (como "El ID es obligatorio") y la traduce a una lista de 700 números. Para la computadora, esa lista de números es el "concepto" de la frase.
Si no estuvieran: No podrías comunicarte con Gemini ni podrías indexar tus documentos para que la IA los entienda.

5. from langchain_core.vectorstores import InMemoryVectorStore
¿Qué es? El "Archivador Inteligente en la Memoria RAM".

¿Para qué sirve? Es una mini base de datos que vive en la memoria de corto plazo de tu computadora (RAM) mientras el programa está encendido. Su única misión es guardar las listas de números (los vectores) que creó el "Traductor Matemático" para que podamos buscar en ellas después.
Si no estuviera: No tendríamos dónde guardar los documentos del negocio indexados, por lo que la IA no sabría dónde buscar las respuestas.

6. from langchain_core.output_parsers import StrOutputParser
¿Qué es? El "Filtro de Limpieza y Desempacado".

¿Para qué sirve? Cuando Gemini responde, no te devuelve solo el texto. Te devuelve un paquete enorme con el ID de la respuesta, cuántos tokens gastó, la hora exacta, el país, etc. StrOutputParser rompe esa caja, tira la basura técnica a la basura y te entrega únicamente el texto limpio de la respuesta.

Si no estuviera: Al hacer un print(), verías una pantalla llena de códigos, llaves y datos técnicos legibles solo para ingenieros.

7. from langchain_core.runnables import RunnablePassthrough
¿Qué es? La "Cinta Transportadora Directa".

¿Para qué sirve? Es un truco de LangChain. Sirve para tomar un dato (como la pregunta que escribió el usuario) y dejarlo pasar hacia el siguiente paso de la tubería sin modificarle absolutamente nada.

Si no estuviera: Tendríamos que crear funciones manuales intermedias solo para mover un dato de un lugar a otro dentro de nuestra cadena.

In [17]:
# Cargar llaves del entorno
load_dotenv()

True

Explicación: Aquí le damos la orden al "Guardia de Seguridad" (load_dotenv) de que empiece a trabajar. Va, lee el archivo .env, encuentra tu API Key y deja la puerta abierta para que Gemini nos reconozca.

In [18]:
print("=== [1] Inicializando Componentes de Gobierno de Datos ===")

=== [1] Inicializando Componentes de Gobierno de Datos ===


In [19]:
# 1. Configuración del Traductor Matemático (Usa el modelo que validaste previamente)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Explicación: Creamos una variable llamada embeddings (una caja con ese nombre) y metemos dentro al traductor oficial de Google usando su modelo más moderno (gemini-embedding-001).

In [20]:
# 2. Base de Conocimiento con Políticas reales de Datos
politicas_gobierno = [
    "Regla DQ-101: El ID de cliente es mandatorio, no puede ser nulo y debe tener 8 dígitos.",
    "Norma REG-SBS: Los reportes de riesgo crediticio deben consolidarse los viernes antes de las 17:00.",
    "Arquitectura Lineage: Todo flujo de datos del Data Lakehouse al catálogo debe usar DataHub.",
    "Política SEC-02: Los datos sensibles (PII) como saldos y contraseñas deben encriptarse en reposo."
]

Explicación: Los corchetes [...] crean una Lista en Python. Es como un cuaderno cuadriculado donde cada fila tiene un texto entre comillas "". Estas son nuestras reglas de negocio.

In [21]:
vectorstore = InMemoryVectorStore.from_texts(politicas_gobierno, embedding=embeddings)
retriever = vectorstore.as_retriever()

InMemoryVectorStore.from_texts(...): Le decimos al Archivador Inteligente: "Toma mi cuaderno de políticas humanas, usa el traductor matemático (embeddings), conviértelo todo a números y guárdalo en la RAM".

vectorstore.as_retriever(): Contratamos a un Bibliotecario (retriever). A partir de ahora, no tocamos la base de datos directamente; le pedimos cosas a este bibliotecario y él sabe cómo buscarlas matemáticamente en el archivador.

In [22]:
# Diseño del Prompt
template = """Actúa como un Oficial de Gobierno de Datos experto. 
Responde la consulta del usuario basándote estrictamente en el contexto normativo proveído.
Si la información no está en el contexto, di amablemente que no cuentas con ese registro en la gobernanza.

Contexto Normativo:
{context}

Consulta: {question}
Respuesta Profesional:"""

Las tres comillas """: Permiten escribir un texto largo con saltos de línea (Enters) sin que Python piense que el código terminó.

Las llaves {context} y {question}: Son marcadores de posición. Están esperando que la tubería les inyecte datos reales ahí dentro.

In [23]:
prompt = ChatPromptTemplate.from_template(template)
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2)

prompt: Mete el texto largo dentro del "Molde Industrial".

model: Despierta al cerebro de Gemini 2.5 Flash Lite. El parámetro temperature=0.2 le quita lo "poeta" y lo vuelve un auditor serio que no inventará reglas que no existan.

In [24]:
# Construcción de la Cadena LCEL
chain = (
    {
        "context": retriever, 
        "question": RunnablePassthrough()
    }
    | prompt 
    | model 
    | StrOutputParser()
)

¿Qué es el símbolo | (Pipe)? Es la magia de LCEL. Significa: "Conecta la salida de la izquierda con la entrada de la derecha". Es un tubo físico por donde viaja la información.  
El primer bloque {...} (Diccionario): Cuando alguien hace una pregunta, este bloque se activa:
  Envía la pregunta al bibliotecario (retriever), este busca en el archivador y mete el texto encontrado en la etiqueta "context".
  Al mismo tiempo, la cinta transportadora (RunnablePassthrough) agarra la pregunta original del usuario y la mete en la etiqueta "question".
Luego, ese par de datos viaja por el tubo | hacia el prompt, que los acomoda dentro de las llaves correspondientes.
El prompt armado viaja por el siguiente tubo | hacia el cerebro de model (Gemini).
La respuesta de Gemini viaja por el último tubo | hacia el limpiador StrOutputParser. Todo este flujo automatizado se guarda bajo el nombre de chain (Cadena).

In [25]:
# =====================================================================
# Paso 5a: BATCH (Procesamiento masivo en paralelo)
# =====================================================================
print("\n=== [5a] Ejecutando BATCH (Auditoría de múltiples consultas) ===")

consultas_masivas = [
    "¿Qué exige la Regla DQ-101 sobre el ID de cliente?",
    "¿A qué hora se deben entregar los reportes de riesgo crediticio para la SBS?",
    "¿Cuál es la política para los saldos y datos sensibles?"
]


=== [5a] Ejecutando BATCH (Auditoría de múltiples consultas) ===


In [26]:
resultados_batch = chain.batch(consultas_masivas)

.batch(): Significa "procesar en lote". Le damos una lista con dos preguntas corporativas. En lugar de procesar la primera, esperar a Google, y luego procesar la segunda; .batch() abre dos canales paralelos en internet, le envía ambas preguntas a Gemini a la vez y recupera las dos respuestas juntas en una lista llamada resultados_batch.

In [27]:
for i, respuesta in enumerate(resultados_batch):
    print(f"\n📌 Pregunta {i+1}: {consultas_masivas[i]}")
    print(f"💡 Respuesta: {respuesta}")

# =====================================================================
# Paso 5b: STREAM (Efecto máquina de escribir en tiempo real)
# =====================================================================
print("\n=== [5b] Ejecutando STREAM (Respuesta interactiva) ===")

consulta_usuario = "¿Cómo debemos documentar el flujo desde el Lakehouse y qué herramienta usamos?"

print(f"\nConsultando: {consulta_usuario}")
print("Asistente escribiendo: ", end="", flush=True)


📌 Pregunta 1: ¿Qué exige la Regla DQ-101 sobre el ID de cliente?
💡 Respuesta: La Regla DQ-101 exige que el ID de cliente sea mandatorio, no pueda ser nulo y deba tener 8 dígitos.

📌 Pregunta 2: ¿A qué hora se deben entregar los reportes de riesgo crediticio para la SBS?
💡 Respuesta: Los reportes de riesgo crediticio deben consolidarse los viernes antes de las 17:00.

📌 Pregunta 3: ¿Cuál es la política para los saldos y datos sensibles?
💡 Respuesta: Según la Política SEC-02, los datos sensibles (PII) como saldos y contraseñas deben encriptarse en reposo.

=== [5b] Ejecutando STREAM (Respuesta interactiva) ===

Consultando: ¿Cómo debemos documentar el flujo desde el Lakehouse y qué herramienta usamos?
Asistente escribiendo: 

for ... in ...: Es un Bucle (Loop). Le dice a Python: "Repite el código que tengo adentro tantas veces como elementos haya en la lista". Como enviamos 2 preguntas, el bucle se ejecuta 2 veces.

enumerate(): Es un contador automatizado. En la primera vuelta, la variable i vale 0 y la variable respuesta tiene el resultado de la primera pregunta. En la segunda vuelta, i vale 1 y respuesta tiene el resultado de la segunda pregunta.

In [30]:
for pedazo in chain.stream(consulta_usuario):
    print(pedazo, end="", flush=True)
print("\n") 

# =====================================================================
# Paso 5c: ASYNC (Operación no bloqueante para APIs/Web)
# =====================================================================
async def servicio_web_simulado():
    print("=== [5c] Ejecutando ASYNC (Asincronía para Microservicios) ===")
    
    pregunta_async = "¿Qué pasa con las contraseñas según la Política SEC-02?"
    respuesta_async = await chain.ainvoke(pregunta_async)
    
    print(f"\n[API Response] Pregunta: {pregunta_async}")
    print(f"[API Response] Respuesta: {respuesta_async}")

# EN JUPYTER NOTEBOOK: Borra el 'if __name__ ...' y ejecuta la función directamente así:
await servicio_web_simulado()

Según la Arquitectura Lineage, todo flujo de datos del Data Lakehouse al catálogo debe usar DataHub.

=== [5c] Ejecutando ASYNC (Asincronía para Microservicios) ===

[API Response] Pregunta: ¿Qué pasa con las contraseñas según la Política SEC-02?
[API Response] Respuesta: Según la Política SEC-02, los datos sensibles (PII) como las contraseñas deben encriptarse en reposo.


.stream(): En lugar de darnos la respuesta al final cuando termine de procesar todo, abre una canilla abierta. Conforme el servidor de Gemini genera una palabra, nos la envía inmediatamente.

for pedazo in ...: Captura cada una de esas palabras sueltas (chunks) que van llegando por el cable.

end="", flush=True: Normalmente, la instrucción print() escribe un texto y salta a la línea de abajo automáticamente. Al poner end="", obligamos a Python a escribir la siguiente palabra exactamente al lado de la anterior. flush=True obliga a la pantalla a mostrar la palabra de inmediato sin guardarla en el búfer de espera. Esto crea el efecto visual de "máquina de escribir" de ChatGPT o Gemini web.

async def: Es la palabra clave para crear una función (un bloque de instrucciones reutilizable) que sea Asíncrona (multitarea).

await: Es la palabra mágica. Le dice a Python: "Oye, voy a lanzar la cadena usando .ainvoke(). Como esto requiere ir hasta los servidores de Google en internet y va a tardar un momento, pongo una pausa AQUÍ. Mientras esperas que Google me devuelva el resultado, tú (Python) quedas libre para ir a atender cualquier otra celda, código o usuario que esté usando el sistema".

await servicio_web_simulado(): Ejecuta nuestra función multitarea respetando el motor asíncrono que ya tiene integrado tu cuaderno de Jupyter.